In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\data\Telco-Customer-Churn.csv')

In [2]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [3]:
df = df.drop("customerID", axis=1)

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df_encoded = pd.get_dummies(df, drop_first=True)

In [9]:
df_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 31 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   SeniorCitizen                          7043 non-null   int64  
 1   tenure                                 7043 non-null   int64  
 2   MonthlyCharges                         7043 non-null   float64
 3   TotalCharges                           7043 non-null   float64
 4   Churn                                  7043 non-null   int64  
 5   gender_Male                            7043 non-null   bool   
 6   Partner_Yes                            7043 non-null   bool   
 7   Dependents_Yes                         7043 non-null   bool   
 8   PhoneService_Yes                       7043 non-null   bool   
 9   MultipleLines_No phone service         7043 non-null   bool   
 10  MultipleLines_Yes                      7043 non-null   bool   
 11  InternetService

In [4]:
X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]


### Decision Tree

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = make_pipeline(
    StandardScaler(),
    DecisionTreeClassifier(max_depth=5, random_state=42)
)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)


print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))



              precision    recall  f1-score   support

           0       0.83      0.93      0.88      1036
           1       0.70      0.46      0.56       373

    accuracy                           0.81      1409
   macro avg       0.77      0.70      0.72      1409
weighted avg       0.80      0.81      0.79      1409

[[964  72]
 [201 172]]


In [12]:
# Gradient Boosting classifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

pipeline_gb = Pipeline([
    ('scaler', StandardScaler()),
    ('gb', GradientBoostingClassifier(random_state=42))
])

pipeline_gb.fit(X_train, y_train)

# Predictions and probabilities
y_pred_gb = pipeline_gb.predict(X_test)
y_proba_gb = pipeline_gb.predict_proba(X_test)[:, 1]

# Training accuracy
train_acc = accuracy_score(y_train, pipeline_gb.predict(X_train))

# Validation (cross-validated) accuracy and ROC AUC on training data
cv_scores_acc = cross_val_score(pipeline_gb, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
cv_scores_gb = cross_val_score(pipeline_gb, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)

print('Gradient Boosting Results:')
print('Training Accuracy: %.3f' % train_acc)
print('Validation CV Accuracy: %.3f ± %.3f' % (cv_scores_acc.mean(), cv_scores_acc.std()))
print('Test Accuracy:', accuracy_score(y_test, y_pred_gb))
print('Test ROC AUC:', roc_auc_score(y_test, y_proba_gb))
print('\nClassification Report:\n', classification_report(y_test, y_pred_gb))
print('CV ROC AUC: %.3f ± %.3f' % (cv_scores_gb.mean(), cv_scores_gb.std()))

Gradient Boosting Results:
Training Accuracy: 0.826
Validation CV Accuracy: 0.800 ± 0.011
Test Accuracy: 0.8069552874378992
Test ROC AUC: 0.8620907387663419

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.90      0.87      1036
           1       0.67      0.54      0.60       373

    accuracy                           0.81      1409
   macro avg       0.76      0.72      0.74      1409
weighted avg       0.80      0.81      0.80      1409

CV ROC AUC: 0.841 ± 0.010


In [11]:
# XGBoost classifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

pipeline_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

pipeline_xgb.fit(X_train, y_train)

# Predictions and probabilities
y_pred_xgb = pipeline_xgb.predict(X_test)
y_proba_xgb = pipeline_xgb.predict_proba(X_test)[:, 1]

# Training accuracy
train_acc = accuracy_score(y_train, pipeline_xgb.predict(X_train))

# Validation (cross-validated) accuracy and ROC AUC on training data
cv_scores_acc = cross_val_score(pipeline_xgb, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
cv_scores_auc = cross_val_score(pipeline_xgb, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)

print('XGBoost Results:')
print('Training Accuracy: %.3f' % train_acc)
print('Validation CV Accuracy: %.3f ± %.3f' % (cv_scores_acc.mean(), cv_scores_acc.std()))
print('Test Accuracy:', accuracy_score(y_test, y_pred_xgb))
print('Test ROC AUC:', roc_auc_score(y_test, y_proba_xgb))
print('\nClassification Report:\n', classification_report(y_test, y_pred_xgb))
print('CV ROC AUC: %.3f ± %.3f' % (cv_scores_auc.mean(), cv_scores_auc.std()))

c:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:42:58] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Results:
Training Accuracy: 0.936
Validation CV Accuracy: 0.781 ± 0.008
Test Accuracy: 0.7963094393186657
Test ROC AUC: 0.8387461571107684

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.89      0.87      1036
           1       0.64      0.53      0.58       373

    accuracy                           0.80      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409

CV ROC AUC: 0.817 ± 0.012
